# Notebook 15. Cleaned Low-Level Feature and Clustering Decision Notebook

This notebook is the cleaned low-level decision point for the revised subtype workflow.

Why this notebook exists:

- `Notebook 10` showed that the `850 hPa` temperature-gradient field was strongly affected by terrain-linked maxima
- `Notebook 11` showed that dropping `T850` entirely changed the clustering too much
- `Notebook 12` showed that a permissive `surface_pressure >= 900 hPa` screen changed almost nothing
- `Notebook 13` showed that explicitly excluding the Russian coastal terrain-interference band strongly changed the `T850` feature and pushed the solution toward `k = 2`
- the remaining open question is whether the **other low-level clustering inputs** at `925 hPa` are also terrain-sensitive enough to matter for the final grouping framework

What this notebook does:

- keeps the Russian-coastal geographic exclusion idea
- rebuilds the terrain-sensitive low-level clustering inputs with that exclusion applied to the gridded fields before regional summarization
- specifically tests:
  - `front_box_max_temp_gradient_850_tminus12_to_tplus12`
  - `coastal_to_jpcz_mean_divergence_ratio`
  - `sea_of_japan_mean_vorticity_peak_925`
- leaves `hokkaido_min_z850_anomaly_tminus12_to_tplus12` unchanged
- reruns the `k = 2/3/4` clustering comparison from the cleaned feature set
- provides the decision outputs needed before making cleaned composite figures in the next notebook

What this notebook does **not** do:

- it does not rebuild the full composite-map atlas from `Notebook 10`
- it does not do spatial EOFs yet
- it does not do the Cluster-1 quartile analysis or the date-pairing analysis yet

Checkpointing note:

- the cleaned event-level low-level feature table is checkpointed to Drive during the long event loop
- summary CSVs and decision plots are also copied to Drive as soon as they are written


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from itertools import permutations
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from jpcz_catalog.analysis import add_japan_local_time_columns
from jpcz_catalog.config import (
    BoundingBox,
    COASTAL_JAPAN_BOX,
    HOKKAIDO_FRONT_BOX,
    JPCZ_POLYGON_VERTICES,
    OBJECTIVE_SUBTYPE_DOMAIN,
    SEA_OF_JAPAN_BOX,
)
from jpcz_catalog.detect import compute_divergence_field, prepare_detection_geometry
from jpcz_catalog.diagnostics import (
    compute_relative_vorticity_field,
    compute_temperature_gradient_magnitude,
    load_offset_snapshot,
)
from jpcz_catalog.era5 import open_arco_era5
from jpcz_catalog.masks import build_coslat_weights
from jpcz_catalog.subtypes import (
    assign_hierarchical_clusters,
    compute_mean_silhouette_score,
    evaluate_hierarchical_cluster_solutions,
    standardize_feature_table,
)

RUN_EXPORT_DIR_08 = Path("outputs/verification/objective_subtype_runs")
RUN_EXPORT_DIR_08.mkdir(parents=True, exist_ok=True)
SENSITIVITY_EXPORT_DIR = Path("outputs/verification/objective_subtype_low_level_cleaned_sensitivity")
SENSITIVITY_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("outputs/verification/objective_subtype_low_level_cleaned_sensitivity_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

CLUSTERED_K3_PATH = RUN_EXPORT_DIR_08 / "clustered_events_k3.csv"
PRIMARY_CLUSTER_COLUMN = "cluster_ward_3"
BASE_RATIO_COLUMN = "coastal_to_jpcz_mean_divergence_ratio"
CLEANED_RATIO_COLUMN = "coastal_to_jpcz_mean_divergence_ratio_russian_coastal_excluded"
BASE_T850_COLUMN = "front_box_max_temp_gradient_850_tminus12_to_tplus12"
CLEANED_T850_COLUMN = "front_box_max_temp_gradient_850_tminus12_to_tplus12_russian_coastal_excluded"
BASE_SOJ_VORT_COLUMN = "sea_of_japan_mean_vorticity_peak_925"
CLEANED_SOJ_VORT_COLUMN = "sea_of_japan_mean_vorticity_peak_925_russian_coastal_excluded"
UNCHANGED_Z850_COLUMN = "hokkaido_min_z850_anomaly_tminus12_to_tplus12"

BASELINE_FEATURE_COLUMNS = [
    BASE_RATIO_COLUMN,
    UNCHANGED_Z850_COLUMN,
    BASE_T850_COLUMN,
    BASE_SOJ_VORT_COLUMN,
]
CLEANED_FEATURE_COLUMNS = [
    CLEANED_RATIO_COLUMN,
    UNCHANGED_Z850_COLUMN,
    CLEANED_T850_COLUMN,
    CLEANED_SOJ_VORT_COLUMN,
]
CLUSTER_COUNT_OPTIONS = [2, 3, 4]
SAVE_PLOTS = True
ERA5_TIME_CHUNK = 48
FORCE_REBUILD_LOW_LEVEL_CLEANED_FEATURES = False
CHECKPOINT_EVERY_EVENTS = 5
OFFSET_HOURS = (-12, 0, 12)

RUSSIAN_COASTAL_EXCLUSION_BOXES = (
    BoundingBox(lon_min=130.5, lon_max=135.5, lat_min=42.0, lat_max=45.0),
    BoundingBox(lon_min=133.5, lon_max=139.5, lat_min=44.0, lat_max=47.25),
)
RUSSIAN_COASTAL_EXCLUSION_LABELS = (
    "Southwestern Russian coastal mountain strip",
    "Northeastern Russian coastal mountain strip",
)

CLEANED_EVENT_FEATURES_PATH = SENSITIVITY_EXPORT_DIR / "k3_cleaned_low_level_event_features.csv"
CLEANED_EVENT_FEATURES_PARTIAL_PATH = SENSITIVITY_EXPORT_DIR / "k3_cleaned_low_level_event_features_partial.csv"
REGION_OVERLAP_SUMMARY_PATH = SENSITIVITY_EXPORT_DIR / "russian_coastal_region_overlap_summary.csv"
SAMPLE_SANITY_PATH = SENSITIVITY_EXPORT_DIR / "sample_cleaned_low_level_feature_sanity.csv"
FEATURE_COMPARISON_PATH = SENSITIVITY_EXPORT_DIR / "cleaned_low_level_feature_comparison_by_current_cluster.csv"
QUALITY_SCAN_LONG_PATH = SENSITIVITY_EXPORT_DIR / "cleaned_low_level_quality_scan_by_solution.csv"
QUALITY_COMPARISON_PATH = SENSITIVITY_EXPORT_DIR / "cleaned_low_level_quality_scan_comparison.csv"
SOLUTION_SUMMARY_PATH = SENSITIVITY_EXPORT_DIR / "cleaned_low_level_solution_summary.csv"
SWITCHING_PATH = SENSITIVITY_EXPORT_DIR / "cleaned_low_level_switching_summary.csv"
CROSSTAB_PATH = SENSITIVITY_EXPORT_DIR / "cleaned_low_level_cluster_crosstabs.csv"
COUNTS_PATH = SENSITIVITY_EXPORT_DIR / "cleaned_low_level_cluster_counts.csv"
MEDIANS_PATH = SENSITIVITY_EXPORT_DIR / "cleaned_low_level_cluster_medians.csv"
VARIANCE_PATH = SENSITIVITY_EXPORT_DIR / "cleaned_low_level_pca_variance.csv"
LOADINGS_PATH = SENSITIVITY_EXPORT_DIR / "cleaned_low_level_pca_loadings.csv"
DRIVERS_PATH = SENSITIVITY_EXPORT_DIR / "cleaned_low_level_pca_drivers.csv"
CLUSTERED_EVENTS_PATH = SENSITIVITY_EXPORT_DIR / "clustered_events_cleaned_low_level_k2_k3_k4.csv"

SILHOUETTE_PLOT_PATH = PLOT_DIR / "cleaned_low_level_silhouette_comparison.png"
FEATURE_BOXPLOT_PATH = PLOT_DIR / "cleaned_low_level_feature_boxplots_by_current_cluster.png"

FEATURE_LABELS = {
    BASE_RATIO_COLUMN: "Coastal/JPCZ signed-divergence ratio",
    CLEANED_RATIO_COLUMN: "Coastal/JPCZ signed-divergence ratio (Russian coastal exclusion)",
    UNCHANGED_Z850_COLUMN: "Hokkaido minimum z850 anomaly",
    BASE_T850_COLUMN: "Front-box maximum |grad T850|",
    CLEANED_T850_COLUMN: "Front-box maximum |grad T850| (Russian coastal exclusion)",
    BASE_SOJ_VORT_COLUMN: "Sea of Japan mean 925 hPa vorticity",
    CLEANED_SOJ_VORT_COLUMN: "Sea of Japan mean 925 hPa vorticity (Russian coastal exclusion)",
}
FEATURE_UNITS = {
    BASE_RATIO_COLUMN: "unitless",
    CLEANED_RATIO_COLUMN: "unitless",
    UNCHANGED_Z850_COLUMN: "gpm",
    BASE_T850_COLUMN: "K (100 km)^-1",
    CLEANED_T850_COLUMN: "K (100 km)^-1",
    BASE_SOJ_VORT_COLUMN: "1e-5 s^-1",
    CLEANED_SOJ_VORT_COLUMN: "1e-5 s^-1",
}


def maybe_copy_to_drive(path, verbose=True):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return None
    drive_path = Path(DRIVE_OUTPUT_DIR) / path.name
    if Path(path).is_file():
        shutil.copy2(path, drive_path)
        if verbose:
            print("Copied to Drive:", drive_path)
        return drive_path
    return None


def restore_from_drive_cache(path):
    if not PERSIST_OUTPUTS_TO_DRIVE:
        return False
    drive_path = Path(DRIVE_OUTPUT_DIR) / Path(path).name
    if not drive_path.exists():
        return False
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(drive_path, path)
    print("Restored from Drive:", drive_path, "->", path)
    return True


def axis_label(column_name):
    label = FEATURE_LABELS.get(column_name, column_name)
    units = FEATURE_UNITS.get(column_name, "")
    if units and units != "unitless":
        return f"{label}\n[{units}]"
    return label


def cluster_count_table(labels):
    counts = labels.value_counts().sort_index()
    max_count = int(counts.max())
    rows = []
    for cluster_id, n_events in counts.items():
        if n_events == max_count:
            size_rank = 1
            size_descriptor = "largest subgroup"
        elif n_events == int(counts.min()):
            size_rank = int((counts.rank(method="dense", ascending=False)).loc[cluster_id])
            size_descriptor = "smallest subgroup"
        else:
            size_rank = int((counts.rank(method="dense", ascending=False)).loc[cluster_id])
            size_descriptor = "second-largest subgroup" if size_rank == 2 else f"rank {size_rank} subgroup"
        rows.append(
            {
                "cluster_id": int(cluster_id),
                "n_events": int(n_events),
                "size_rank": size_rank,
                "size_descriptor": size_descriptor,
                "cluster_label": f"Cluster {int(cluster_id)}: n={int(n_events)} ({size_descriptor})",
            }
        )
    return pd.DataFrame(rows)


def best_cluster_label_mapping(reference_labels, candidate_labels):
    reference = reference_labels.astype(int)
    candidate = candidate_labels.astype(int).reindex(reference.index)
    unique_reference = sorted(reference.dropna().unique().tolist())
    unique_candidate = sorted(candidate.dropna().unique().tolist())
    if len(unique_reference) != len(unique_candidate):
        raise ValueError(
            "Expected the same number of clusters in the reference and candidate solutions, "
            f"got {unique_reference} versus {unique_candidate}."
        )

    best_mapping = None
    best_matches = -1
    for perm in permutations(unique_reference, len(unique_candidate)):
        mapping = dict(zip(unique_candidate, perm))
        mapped = candidate.map(mapping)
        match_count = int((mapped == reference).sum())
        if match_count > best_matches:
            best_matches = match_count
            best_mapping = mapping
    return best_mapping, best_matches


def compute_pca_diagnostics(feature_df, feature_columns):
    standardized_df, feature_means, feature_stds = standardize_feature_table(feature_df.copy(), columns=feature_columns)
    valid = standardized_df.dropna(axis=0, how="any")
    if valid.empty:
        raise ValueError("No complete rows are available for PCA.")

    matrix = valid.to_numpy(dtype=float)
    _, singular_values, vt = np.linalg.svd(matrix, full_matrices=False)
    n_components = min(3, matrix.shape[1])
    components = vt[:n_components]
    scores = matrix @ components.T

    total_variance = float((singular_values ** 2).sum())
    explained_variance_ratio = (singular_values[:n_components] ** 2) / total_variance

    score_df = pd.DataFrame(
        scores,
        index=valid.index,
        columns=[f"PC{i + 1}" for i in range(n_components)],
    )
    variance_df = pd.DataFrame(
        {
            "principal_component": [f"PC{i + 1}" for i in range(n_components)],
            "explained_variance_ratio": explained_variance_ratio,
            "explained_variance_percent": explained_variance_ratio * 100.0,
            "cumulative_explained_variance_ratio": np.cumsum(explained_variance_ratio),
            "cumulative_explained_variance_percent": np.cumsum(explained_variance_ratio) * 100.0,
        }
    )
    loadings_df = pd.DataFrame(
        components.T,
        index=feature_columns,
        columns=[f"PC{i + 1}" for i in range(n_components)],
    ).reset_index().rename(columns={"index": "feature"})
    loadings_long_df = loadings_df.melt(id_vars="feature", var_name="principal_component", value_name="loading")
    loadings_long_df["feature_label"] = loadings_long_df["feature"].map(FEATURE_LABELS)

    driver_rows = []
    for pc_name in variance_df["principal_component"]:
        component_subset = loadings_long_df.loc[loadings_long_df["principal_component"] == pc_name].copy()
        component_subset["abs_loading"] = component_subset["loading"].abs()
        top_two = component_subset.sort_values("abs_loading", ascending=False).head(2).reset_index(drop=True)
        driver_rows.append(
            {
                "principal_component": pc_name,
                "explained_variance_percent": float(
                    variance_df.loc[variance_df["principal_component"] == pc_name, "explained_variance_percent"].iloc[0]
                ),
                "top_driver_feature": top_two.loc[0, "feature"],
                "top_driver_label": top_two.loc[0, "feature_label"],
                "top_driver_loading": float(top_two.loc[0, "loading"]),
                "second_driver_feature": top_two.loc[1, "feature"] if len(top_two) > 1 else "",
                "second_driver_label": top_two.loc[1, "feature_label"] if len(top_two) > 1 else "",
                "second_driver_loading": float(top_two.loc[1, "loading"]) if len(top_two) > 1 else np.nan,
            }
        )
    driver_df = pd.DataFrame(driver_rows)
    return standardized_df, score_df, variance_df, loadings_long_df, driver_df, feature_means, feature_stds


def safe_ratio(numerator, denominator):
    if not np.isfinite(numerator) or not np.isfinite(denominator) or denominator == 0.0:
        return float("nan")
    return float(numerator / denominator)


def safe_nanmax(values):
    array = np.asarray(values, dtype=float)
    if array.size == 0 or np.all(np.isnan(array)):
        return float("nan")
    return float(np.nanmax(array))


def subset_to_box(field, box):
    return field.sel(
        longitude=slice(box.lon_min, box.lon_max),
        latitude=slice(box.lat_max, box.lat_min),
    )


def exclusion_box_table():
    rows = []
    for label, box in zip(RUSSIAN_COASTAL_EXCLUSION_LABELS, RUSSIAN_COASTAL_EXCLUSION_BOXES):
        rows.append(
            {
                "label": label,
                "lon_min": box.lon_min,
                "lon_max": box.lon_max,
                "lat_min": box.lat_min,
                "lat_max": box.lat_max,
            }
        )
    return pd.DataFrame(rows)


def build_russian_coastal_keep_mask(target_field):
    lat_vals = np.asarray(target_field.latitude.values, dtype=float)
    lon_vals = np.asarray(target_field.longitude.values, dtype=float)
    lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)
    keep_mask = np.ones((len(lat_vals), len(lon_vals)), dtype=bool)

    for box in RUSSIAN_COASTAL_EXCLUSION_BOXES:
        in_box = (
            (lon2d >= box.lon_min)
            & (lon2d <= box.lon_max)
            & (lat2d >= box.lat_min)
            & (lat2d <= box.lat_max)
        )
        keep_mask &= ~in_box

    mask = xr.DataArray(
        keep_mask,
        coords={"latitude": target_field.latitude, "longitude": target_field.longitude},
        dims=("latitude", "longitude"),
        name="russian_coastal_keep_mask",
    )
    mask.attrs["meaning"] = "True outside the Russian coastal terrain-interference exclusion boxes"
    return mask


def weighted_mean_with_keep_mask(field, keep_mask, extra_mask=None):
    valid_mask = keep_mask.copy()
    if extra_mask is not None:
        valid_mask = valid_mask & extra_mask
    valid_mask = valid_mask & field.notnull()
    weights = build_coslat_weights(field.latitude, field.longitude)
    valid_weights = weights.where(valid_mask, 0.0)
    total_weight = float(valid_weights.sum().values)
    if total_weight == 0.0:
        return float("nan")
    return float(((field.fillna(0.0) * valid_weights).sum() / total_weight).values)


def box_weighted_mean_with_keep_mask(field, box, keep_mask):
    boxed = subset_to_box(field, box)
    boxed_keep = subset_to_box(keep_mask, box)
    return weighted_mean_with_keep_mask(boxed, boxed_keep)


def polygon_weighted_mean_with_keep_mask(field, polygon_mask, keep_mask):
    return weighted_mean_with_keep_mask(field, keep_mask, extra_mask=polygon_mask)


def box_max_with_keep_mask(field, box, keep_mask):
    boxed = subset_to_box(field, box)
    boxed_keep = subset_to_box(keep_mask, box)
    masked = boxed.where(boxed_keep)
    return float(masked.max(skipna=True).values)


def box_keep_fraction(keep_mask, box):
    boxed = subset_to_box(keep_mask, box)
    return float(boxed.mean().values)


def polygon_keep_fraction(keep_mask, polygon_mask):
    masked = keep_mask.where(polygon_mask)
    return float(masked.mean(skipna=True).values)


In [ ]:
for path_to_restore in [CLUSTERED_K3_PATH]:
    if not path_to_restore.exists():
        restore_from_drive_cache(CLUSTERED_K3_PATH)

clustered_k3_df = pd.read_csv(CLUSTERED_K3_PATH)
clustered_k3_df = add_japan_local_time_columns(clustered_k3_df)
if PRIMARY_CLUSTER_COLUMN not in clustered_k3_df.columns:
    cluster_cols = [column for column in clustered_k3_df.columns if column.startswith("cluster_")]
    raise RuntimeError(f"Expected {PRIMARY_CLUSTER_COLUMN} in clustered_events_k3.csv, found {cluster_cols}")

required_columns = BASELINE_FEATURE_COLUMNS
missing_columns = [column for column in required_columns if column not in clustered_k3_df.columns]
if missing_columns:
    raise RuntimeError(f"Missing required baseline feature columns: {missing_columns}")

current_cluster_labels = clustered_k3_df[PRIMARY_CLUSTER_COLUMN].astype(int)
current_cluster_counts_df = cluster_count_table(current_cluster_labels)
current_cluster_label_lookup = current_cluster_counts_df.set_index("cluster_id")["cluster_label"].to_dict()

print("Loaded current clustered k=3 rerun table from Notebook 08 outputs")
display(current_cluster_counts_df)
print("\nNotebook 15 keeps the baseline feature table but rebuilds the terrain-sensitive low-level inputs with the Russian-coastal exclusion applied first.")


In [ ]:
if 'era5_runtime_ds' not in globals():
    era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})

sample_event_row = clustered_k3_df.iloc[0].copy()

sample_snapshot_925 = load_offset_snapshot(
    era5_runtime_ds,
    sample_event_row["event_peak"],
    offset_hours=0,
    variables=["u_component_of_wind", "v_component_of_wind"],
    domain=OBJECTIVE_SUBTYPE_DOMAIN,
    level=925,
)
geometry_925_sample = prepare_detection_geometry(
    sample_snapshot_925.longitude,
    sample_snapshot_925.latitude,
    JPCZ_POLYGON_VERTICES,
)
sample_divergence_925 = (compute_divergence_field(sample_snapshot_925, dx=geometry_925_sample.dx, dy=geometry_925_sample.dy) * 1e5).rename("divergence_925_display")
sample_keep_mask_925 = build_russian_coastal_keep_mask(sample_divergence_925)

sample_snapshot_850 = load_offset_snapshot(
    era5_runtime_ds,
    sample_event_row["event_peak"],
    offset_hours=0,
    variables=["temperature"],
    domain=OBJECTIVE_SUBTYPE_DOMAIN,
    level=850,
)
sample_temp_grad = compute_temperature_gradient_magnitude(sample_snapshot_850)
sample_temp_grad_display = (sample_temp_grad * float(sample_temp_grad.attrs.get("display_scale_factor", 1.0))).rename("temperature_gradient_display")
sample_keep_mask_850 = build_russian_coastal_keep_mask(sample_temp_grad_display)

region_overlap_summary_df = pd.DataFrame(
    [
        {
            "field_grid": "925 hPa low-level grid",
            "region": "Full domain",
            "keep_fraction": round(float(sample_keep_mask_925.mean().values), 3),
        },
        {
            "field_grid": "925 hPa low-level grid",
            "region": "JPCZ polygon",
            "keep_fraction": round(polygon_keep_fraction(sample_keep_mask_925, geometry_925_sample.polygon_mask), 3),
        },
        {
            "field_grid": "925 hPa low-level grid",
            "region": "Coastal Japan box",
            "keep_fraction": round(box_keep_fraction(sample_keep_mask_925, COASTAL_JAPAN_BOX), 3),
        },
        {
            "field_grid": "925 hPa low-level grid",
            "region": "Sea of Japan box",
            "keep_fraction": round(box_keep_fraction(sample_keep_mask_925, SEA_OF_JAPAN_BOX), 3),
        },
        {
            "field_grid": "850 hPa frontality grid",
            "region": "Full domain",
            "keep_fraction": round(float(sample_keep_mask_850.mean().values), 3),
        },
        {
            "field_grid": "850 hPa frontality grid",
            "region": "Hokkaido front box",
            "keep_fraction": round(box_keep_fraction(sample_keep_mask_850, HOKKAIDO_FRONT_BOX), 3),
        },
    ]
)
region_overlap_summary_df.to_csv(REGION_OVERLAP_SUMMARY_PATH, index=False)
maybe_copy_to_drive(REGION_OVERLAP_SUMMARY_PATH)

print("Russian-coastal low-level exclusion overlap summary")
display(region_overlap_summary_df)
print("\nRussian coastal exclusion boxes")
display(exclusion_box_table())


In [ ]:
if 'region_overlap_summary_df' not in globals() or 'clustered_k3_df' not in globals():
    raise RuntimeError(
        "Run the region-overlap summary cell first so the exclusion geometry and clustered table are defined before the sanity check."
    )

if 'era5_runtime_ds' not in globals():
    era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})

sample_event_row = clustered_k3_df.iloc[0].copy()

sample_snapshot_925 = load_offset_snapshot(
    era5_runtime_ds,
    sample_event_row["event_peak"],
    offset_hours=0,
    variables=["u_component_of_wind", "v_component_of_wind"],
    domain=OBJECTIVE_SUBTYPE_DOMAIN,
    level=925,
)
geometry_925_sample = prepare_detection_geometry(
    sample_snapshot_925.longitude,
    sample_snapshot_925.latitude,
    JPCZ_POLYGON_VERTICES,
)
sample_divergence_925 = (compute_divergence_field(sample_snapshot_925, dx=geometry_925_sample.dx, dy=geometry_925_sample.dy) * 1e5).rename("divergence_925_display")
sample_vorticity_925 = (compute_relative_vorticity_field(sample_snapshot_925, dx=geometry_925_sample.dx, dy=geometry_925_sample.dy) * 1e5).rename("vorticity_925_display")
sample_keep_mask_925 = build_russian_coastal_keep_mask(sample_divergence_925)

sample_cleaned_jpcz_mean = polygon_weighted_mean_with_keep_mask(sample_divergence_925, geometry_925_sample.polygon_mask, sample_keep_mask_925)
sample_cleaned_coastal_mean = box_weighted_mean_with_keep_mask(sample_divergence_925, COASTAL_JAPAN_BOX, sample_keep_mask_925)
sample_cleaned_ratio = safe_ratio(sample_cleaned_coastal_mean, sample_cleaned_jpcz_mean)
sample_cleaned_soj_vort = box_weighted_mean_with_keep_mask(sample_vorticity_925, SEA_OF_JAPAN_BOX, sample_keep_mask_925)

sample_cleaned_t850_values = []
for offset_hours in OFFSET_HOURS:
    sample_snapshot_850 = load_offset_snapshot(
        era5_runtime_ds,
        sample_event_row["event_peak"],
        offset_hours=offset_hours,
        variables=["temperature"],
        domain=OBJECTIVE_SUBTYPE_DOMAIN,
        level=850,
    )
    sample_temp_grad = compute_temperature_gradient_magnitude(sample_snapshot_850)
    sample_temp_grad_display = (sample_temp_grad * float(sample_temp_grad.attrs.get("display_scale_factor", 1.0))).rename("temperature_gradient_display")
    sample_keep_mask_850 = build_russian_coastal_keep_mask(sample_temp_grad_display)
    sample_cleaned_t850_values.append(box_max_with_keep_mask(sample_temp_grad_display, HOKKAIDO_FRONT_BOX, sample_keep_mask_850))

sample_sanity_df = pd.DataFrame(
    {
        "metric": [
            "sample event peak",
            "sample current cluster",
            "925 JPCZ polygon keep fraction",
            "925 Coastal Japan box keep fraction",
            "925 Sea of Japan box keep fraction",
            "850 Hokkaido front box keep fraction at t0",
            "sample unmasked coastal/JPCZ ratio",
            "sample cleaned coastal/JPCZ ratio",
            "sample unmasked Sea of Japan mean 925 hPa vorticity [1e-5 s^-1]",
            "sample cleaned Sea of Japan mean 925 hPa vorticity [1e-5 s^-1]",
            "sample unmasked event-level T850 feature [K (100 km)^-1]",
            "sample cleaned event-level T850 feature [K (100 km)^-1]",
        ],
        "value": [
            str(pd.Timestamp(sample_event_row["event_peak"])),
            int(sample_event_row[PRIMARY_CLUSTER_COLUMN]),
            round(polygon_keep_fraction(sample_keep_mask_925, geometry_925_sample.polygon_mask), 3),
            round(box_keep_fraction(sample_keep_mask_925, COASTAL_JAPAN_BOX), 3),
            round(box_keep_fraction(sample_keep_mask_925, SEA_OF_JAPAN_BOX), 3),
            round(box_keep_fraction(sample_keep_mask_850, HOKKAIDO_FRONT_BOX), 3),
            round(float(sample_event_row[BASE_RATIO_COLUMN]), 3),
            round(float(sample_cleaned_ratio), 3),
            round(float(sample_event_row[BASE_SOJ_VORT_COLUMN]), 3),
            round(float(sample_cleaned_soj_vort), 3),
            round(float(sample_event_row[BASE_T850_COLUMN]), 3),
            round(float(safe_nanmax(sample_cleaned_t850_values)), 3),
        ],
    }
)
sample_sanity_df.to_csv(SAMPLE_SANITY_PATH, index=False)
maybe_copy_to_drive(SAMPLE_SANITY_PATH)

print("One-event Russian-coastal cleaned low-level feature sanity check before the full event loop")
display(sample_sanity_df)


In [ ]:
for path_to_restore in [CLEANED_EVENT_FEATURES_PATH, CLEANED_EVENT_FEATURES_PARTIAL_PATH]:
    if not path_to_restore.exists():
        restore_from_drive_cache(path_to_restore)

if 'era5_runtime_ds' not in globals():
    era5_runtime_ds = open_arco_era5(chunks={"time": ERA5_TIME_CHUNK})

existing_cleaned_df = None
if CLEANED_EVENT_FEATURES_PATH.exists() and not FORCE_REBUILD_LOW_LEVEL_CLEANED_FEATURES:
    existing_cleaned_df = pd.read_csv(CLEANED_EVENT_FEATURES_PATH)
    print("Loaded cached final cleaned low-level event feature table")
elif CLEANED_EVENT_FEATURES_PARTIAL_PATH.exists() and not FORCE_REBUILD_LOW_LEVEL_CLEANED_FEATURES:
    existing_cleaned_df = pd.read_csv(CLEANED_EVENT_FEATURES_PARTIAL_PATH)
    print("Loaded cached partial cleaned low-level event feature table")

cleaned_feature_columns = [CLEANED_RATIO_COLUMN, CLEANED_T850_COLUMN, CLEANED_SOJ_VORT_COLUMN]
if existing_cleaned_df is not None and not existing_cleaned_df.empty:
    existing_cleaned_df["event_row_index"] = pd.to_numeric(existing_cleaned_df["event_row_index"], errors="coerce").astype("Int64")
    for column_name in cleaned_feature_columns:
        existing_cleaned_df[column_name] = pd.to_numeric(existing_cleaned_df[column_name], errors="coerce")
    finite_cached = int(existing_cleaned_df[cleaned_feature_columns].notna().any(axis=1).sum())
    print(f"Cached cleaned low-level rows with at least one finite cleaned feature value: {finite_cached}/{len(existing_cleaned_df)}")
    if finite_cached == 0:
        print("Cached cleaned low-level feature table has zero finite values, so it will be ignored and rebuilt.")
        existing_cleaned_df = None

processed_records = []
processed_event_indices = set()
if existing_cleaned_df is not None and not existing_cleaned_df.empty:
    processed_records = existing_cleaned_df.to_dict(orient="records")
    processed_event_indices = {int(value) for value in existing_cleaned_df["event_row_index"].dropna().tolist()}

geometry_925 = None
keep_mask_925_template = None
keep_mask_850_template = None


def compute_cleaned_low_level_features_for_event(row):
    global geometry_925, keep_mask_925_template, keep_mask_850_template

    peak_snapshot_925 = load_offset_snapshot(
        era5_runtime_ds,
        row["event_peak"],
        offset_hours=0,
        variables=["u_component_of_wind", "v_component_of_wind"],
        domain=OBJECTIVE_SUBTYPE_DOMAIN,
        level=925,
    )
    if geometry_925 is None:
        geometry_925 = prepare_detection_geometry(
            peak_snapshot_925.longitude,
            peak_snapshot_925.latitude,
            JPCZ_POLYGON_VERTICES,
        )

    divergence_field = (compute_divergence_field(peak_snapshot_925, dx=geometry_925.dx, dy=geometry_925.dy) * 1e5).rename("divergence_925_display")
    vorticity_field = (compute_relative_vorticity_field(peak_snapshot_925, dx=geometry_925.dx, dy=geometry_925.dy) * 1e5).rename("vorticity_925_display")
    if keep_mask_925_template is None or keep_mask_925_template.shape != divergence_field.shape:
        keep_mask_925_template = build_russian_coastal_keep_mask(divergence_field)

    cleaned_jpcz_mean = polygon_weighted_mean_with_keep_mask(divergence_field, geometry_925.polygon_mask, keep_mask_925_template)
    cleaned_coastal_mean = box_weighted_mean_with_keep_mask(divergence_field, COASTAL_JAPAN_BOX, keep_mask_925_template)
    cleaned_ratio = safe_ratio(cleaned_coastal_mean, cleaned_jpcz_mean)
    cleaned_soj_vort = box_weighted_mean_with_keep_mask(vorticity_field, SEA_OF_JAPAN_BOX, keep_mask_925_template)

    cleaned_t850_values = []
    for offset_hours in OFFSET_HOURS:
        snapshot_850 = load_offset_snapshot(
            era5_runtime_ds,
            row["event_peak"],
            offset_hours=offset_hours,
            variables=["temperature"],
            domain=OBJECTIVE_SUBTYPE_DOMAIN,
            level=850,
        )
        temp_grad = compute_temperature_gradient_magnitude(snapshot_850)
        temp_grad_display = (temp_grad * float(temp_grad.attrs.get("display_scale_factor", 1.0))).rename("temperature_gradient_display")
        if keep_mask_850_template is None or keep_mask_850_template.shape != temp_grad_display.shape:
            keep_mask_850_template = build_russian_coastal_keep_mask(temp_grad_display)
        cleaned_t850_values.append(box_max_with_keep_mask(temp_grad_display, HOKKAIDO_FRONT_BOX, keep_mask_850_template))

    return {
        CLEANED_RATIO_COLUMN: cleaned_ratio,
        CLEANED_SOJ_VORT_COLUMN: cleaned_soj_vort,
        CLEANED_T850_COLUMN: safe_nanmax(cleaned_t850_values),
    }

remaining_rows = clustered_k3_df.loc[~clustered_k3_df.index.isin(processed_event_indices)].copy()
print(f"Cleaned low-level feature progress before this run: {len(processed_event_indices)}/{len(clustered_k3_df)} events")

for _, (row_index, row) in enumerate(remaining_rows.iterrows(), start=1):
    cleaned_values = compute_cleaned_low_level_features_for_event(row)
    processed_records.append(
        {
            "event_row_index": int(row_index),
            "event_peak": pd.Timestamp(row["event_peak"]),
            "current_cluster_id": int(row[PRIMARY_CLUSTER_COLUMN]),
            "current_cluster_label": current_cluster_label_lookup[int(row[PRIMARY_CLUSTER_COLUMN])],
            BASE_RATIO_COLUMN: float(row[BASE_RATIO_COLUMN]),
            CLEANED_RATIO_COLUMN: cleaned_values[CLEANED_RATIO_COLUMN],
            "cleaned_minus_unmasked_ratio": cleaned_values[CLEANED_RATIO_COLUMN] - float(row[BASE_RATIO_COLUMN]) if np.isfinite(cleaned_values[CLEANED_RATIO_COLUMN]) else np.nan,
            BASE_SOJ_VORT_COLUMN: float(row[BASE_SOJ_VORT_COLUMN]),
            CLEANED_SOJ_VORT_COLUMN: cleaned_values[CLEANED_SOJ_VORT_COLUMN],
            "cleaned_minus_unmasked_soj_vorticity": cleaned_values[CLEANED_SOJ_VORT_COLUMN] - float(row[BASE_SOJ_VORT_COLUMN]) if np.isfinite(cleaned_values[CLEANED_SOJ_VORT_COLUMN]) else np.nan,
            BASE_T850_COLUMN: float(row[BASE_T850_COLUMN]),
            CLEANED_T850_COLUMN: cleaned_values[CLEANED_T850_COLUMN],
            "cleaned_minus_unmasked_t850": cleaned_values[CLEANED_T850_COLUMN] - float(row[BASE_T850_COLUMN]) if np.isfinite(cleaned_values[CLEANED_T850_COLUMN]) else np.nan,
        }
    )

    event_number = len(processed_records)
    if event_number % CHECKPOINT_EVERY_EVENTS == 0 or event_number == len(clustered_k3_df):
        checkpoint_df = pd.DataFrame(processed_records).sort_values("event_row_index").reset_index(drop=True)
        checkpoint_df.to_csv(CLEANED_EVENT_FEATURES_PARTIAL_PATH, index=False)
        maybe_copy_to_drive(CLEANED_EVENT_FEATURES_PARTIAL_PATH)
        print(f"Checkpointed cleaned low-level event feature table at {event_number}/{len(clustered_k3_df)} events")

cleaned_event_feature_df = pd.DataFrame(processed_records).sort_values("event_row_index").reset_index(drop=True)
cleaned_event_feature_df["event_row_index"] = pd.to_numeric(cleaned_event_feature_df["event_row_index"], errors="coerce").astype(int)
for column_name in cleaned_feature_columns:
    cleaned_event_feature_df[column_name] = pd.to_numeric(cleaned_event_feature_df[column_name], errors="coerce")
finite_cleaned_events = int(cleaned_event_feature_df[cleaned_feature_columns].notna().all(axis=1).sum())
print(f"Rows with all three finite cleaned low-level features after this run: {finite_cleaned_events}/{len(cleaned_event_feature_df)} events")
if finite_cleaned_events == 0:
    raise RuntimeError(
        "The cleaned low-level feature table contains zero rows with all three finite cleaned features. "
        "Do not continue until the exclusion design is corrected."
    )
cleaned_event_feature_df.to_csv(CLEANED_EVENT_FEATURES_PATH, index=False)
maybe_copy_to_drive(CLEANED_EVENT_FEATURES_PATH)
print("Saved final cleaned low-level event feature table")

comparison_specs = [
    (BASE_RATIO_COLUMN, CLEANED_RATIO_COLUMN, "Coastal/JPCZ signed-divergence ratio"),
    (BASE_SOJ_VORT_COLUMN, CLEANED_SOJ_VORT_COLUMN, "Sea of Japan mean 925 hPa vorticity"),
    (BASE_T850_COLUMN, CLEANED_T850_COLUMN, "Front-box maximum |grad T850|"),
]
feature_comparison_rows = []
for cluster_id, subset in cleaned_event_feature_df.groupby("current_cluster_id"):
    for base_column, cleaned_column, feature_label in comparison_specs:
        delta = subset[cleaned_column] - subset[base_column]
        feature_comparison_rows.append(
            {
                "feature_label": feature_label,
                "base_column": base_column,
                "cleaned_column": cleaned_column,
                "cluster_id": int(cluster_id),
                "cluster_label": subset["current_cluster_label"].iloc[0],
                "n_events": int(len(subset)),
                "unmasked_median": float(subset[base_column].median()),
                "cleaned_median": float(subset[cleaned_column].median()),
                "median_cleaned_minus_unmasked": float(delta.median()),
                "mean_abs_cleaned_minus_unmasked": float(np.nanmean(np.abs(delta))),
            }
        )
feature_comparison_df = pd.DataFrame(feature_comparison_rows)
for column_name in ["unmasked_median", "cleaned_median", "median_cleaned_minus_unmasked", "mean_abs_cleaned_minus_unmasked"]:
    feature_comparison_df[column_name] = feature_comparison_df[column_name].round(3)
feature_comparison_df.to_csv(FEATURE_COMPARISON_PATH, index=False)
maybe_copy_to_drive(FEATURE_COMPARISON_PATH)

print("Per-current-cluster comparison of unmasked versus cleaned low-level features")
display(feature_comparison_df)


In [ ]:
baseline_standardized_df, baseline_scores_df, baseline_variance_df, baseline_loadings_df, baseline_driver_df, baseline_feature_means, baseline_feature_stds = compute_pca_diagnostics(
    clustered_k3_df,
    BASELINE_FEATURE_COLUMNS,
)

cleaned_feature_df = clustered_k3_df.copy()
cleaned_event_feature_df["event_row_index"] = pd.to_numeric(cleaned_event_feature_df["event_row_index"], errors="coerce").astype(int)
cleaned_lookup = cleaned_event_feature_df.set_index("event_row_index")[[CLEANED_RATIO_COLUMN, CLEANED_T850_COLUMN, CLEANED_SOJ_VORT_COLUMN]]
for column_name in [CLEANED_RATIO_COLUMN, CLEANED_T850_COLUMN, CLEANED_SOJ_VORT_COLUMN]:
    cleaned_feature_df[column_name] = pd.to_numeric(cleaned_lookup[column_name].reindex(cleaned_feature_df.index), errors="coerce")

finite_cleaned_feature_rows = int(cleaned_feature_df[CLEANED_FEATURE_COLUMNS].notna().all(axis=1).sum())
print(f"Rows with finite cleaned low-level feature sets entering PCA/clustering: {finite_cleaned_feature_rows}/{len(cleaned_feature_df)}")
if finite_cleaned_feature_rows == 0:
    raise RuntimeError(
        "No complete cleaned low-level feature rows are available for PCA/clustering. "
        "Rerun the summary, sanity-check, and cleaned-feature build cells with the updated logic."
    )

cleaned_standardized_df, cleaned_scores_df, cleaned_variance_df, cleaned_loadings_df, cleaned_driver_df, cleaned_feature_means, cleaned_feature_stds = compute_pca_diagnostics(
    cleaned_feature_df,
    CLEANED_FEATURE_COLUMNS,
)

baseline_quality_df = evaluate_hierarchical_cluster_solutions(
    baseline_standardized_df,
    cluster_counts=CLUSTER_COUNT_OPTIONS,
    method="ward",
).assign(solution="baseline_current_features")
cleaned_quality_df = evaluate_hierarchical_cluster_solutions(
    cleaned_standardized_df,
    cluster_counts=CLUSTER_COUNT_OPTIONS,
    method="ward",
).assign(solution="russian_coastal_cleaned_low_level")
quality_scan_long_df = pd.concat([baseline_quality_df, cleaned_quality_df], ignore_index=True)
quality_comparison_df = baseline_quality_df.merge(
    cleaned_quality_df,
    on="n_clusters",
    suffixes=("_baseline", "_cleaned"),
)
quality_comparison_df = quality_comparison_df[[
    "n_clusters",
    "mean_silhouette_score_baseline",
    "mean_silhouette_score_cleaned",
    "smallest_cluster_size_baseline",
    "smallest_cluster_size_cleaned",
    "largest_cluster_size_baseline",
    "largest_cluster_size_cleaned",
]]
for column_name in ["mean_silhouette_score_baseline", "mean_silhouette_score_cleaned"]:
    quality_comparison_df[column_name] = quality_comparison_df[column_name].round(5)
quality_scan_long_df.to_csv(QUALITY_SCAN_LONG_PATH, index=False)
maybe_copy_to_drive(QUALITY_SCAN_LONG_PATH)
quality_comparison_df.to_csv(QUALITY_COMPARISON_PATH, index=False)
maybe_copy_to_drive(QUALITY_COMPARISON_PATH)

recommended_k = int(cleaned_quality_df.sort_values(["mean_silhouette_score", "smallest_cluster_size"], ascending=[False, False]).iloc[0]["n_clusters"])
print(f"Recommended cleaned cluster count from the silhouette scan: k={recommended_k}")

switching_rows = []
crosstab_rows = []
count_rows = []
median_rows = []
solution_rows = []
clustered_events_export_df = cleaned_feature_df.copy()

variance_export_df = pd.concat(
    [
        baseline_variance_df.assign(solution="baseline_current_features"),
        cleaned_variance_df.assign(solution="russian_coastal_cleaned_low_level"),
    ],
    ignore_index=True,
)
loadings_export_df = pd.concat(
    [
        baseline_loadings_df.assign(solution="baseline_current_features"),
        cleaned_loadings_df.assign(solution="russian_coastal_cleaned_low_level"),
    ],
    ignore_index=True,
)
driver_export_df = pd.concat(
    [
        baseline_driver_df.assign(solution="baseline_current_features"),
        cleaned_driver_df.assign(solution="russian_coastal_cleaned_low_level"),
    ],
    ignore_index=True,
)

for k in CLUSTER_COUNT_OPTIONS:
    baseline_labels = assign_hierarchical_clusters(
        baseline_standardized_df,
        n_clusters=k,
        method="ward",
    ).rename(f"baseline_cluster_ward_{k}")
    cleaned_labels_raw = assign_hierarchical_clusters(
        cleaned_standardized_df,
        n_clusters=k,
        method="ward",
    ).rename(f"cleaned_cluster_ward_{k}_raw")
    label_mapping, best_match_count = best_cluster_label_mapping(baseline_labels, cleaned_labels_raw)
    cleaned_labels = cleaned_labels_raw.map(label_mapping).rename(f"cleaned_cluster_ward_{k}")
    clustered_events_export_df[cleaned_labels_raw.name] = cleaned_labels_raw.reindex(clustered_events_export_df.index)
    clustered_events_export_df[cleaned_labels.name] = cleaned_labels.reindex(clustered_events_export_df.index)

    baseline_counts_df = cluster_count_table(baseline_labels)
    cleaned_counts_df = cluster_count_table(cleaned_labels)
    baseline_silhouette = float(compute_mean_silhouette_score(baseline_standardized_df, baseline_labels))
    cleaned_silhouette = float(compute_mean_silhouette_score(cleaned_standardized_df, cleaned_labels))

    solution_rows.extend(
        [
            {
                "solution": "baseline_current_features",
                "n_clusters": int(k),
                "feature_columns": ", ".join(BASELINE_FEATURE_COLUMNS),
                "silhouette": round(baseline_silhouette, 5),
                "cluster_counts": ", ".join([f"{int(row.cluster_id)}:{int(row.n_events)}" for row in baseline_counts_df.itertuples(index=False)]),
                "recommended_cleaned_solution": False,
            },
            {
                "solution": "russian_coastal_cleaned_low_level",
                "n_clusters": int(k),
                "feature_columns": ", ".join(CLEANED_FEATURE_COLUMNS),
                "silhouette": round(cleaned_silhouette, 5),
                "cluster_counts": ", ".join([f"{int(row.cluster_id)}:{int(row.n_events)}" for row in cleaned_counts_df.itertuples(index=False)]),
                "recommended_cleaned_solution": bool(k == recommended_k),
            },
        ]
    )

    switch_mask = cleaned_labels != baseline_labels.loc[cleaned_labels.index]
    switching_rows.append(
        {
            "n_clusters": int(k),
            "cluster_id": "all",
            "cluster_label": f"All events (baseline k={k})",
            "n_events": int(len(cleaned_labels)),
            "n_switched": int(switch_mask.sum()),
            "percent_switched": round(float(100.0 * switch_mask.mean()), 2),
            "best_label_mapping": str(label_mapping),
            "best_label_match_count": int(best_match_count),
        }
    )
    baseline_label_lookup = baseline_counts_df.set_index("cluster_id")["cluster_label"].to_dict()
    for cluster_id in sorted(baseline_labels.dropna().astype(int).unique()):
        cluster_mask = baseline_labels == int(cluster_id)
        switching_rows.append(
            {
                "n_clusters": int(k),
                "cluster_id": int(cluster_id),
                "cluster_label": baseline_label_lookup[int(cluster_id)],
                "n_events": int(cluster_mask.sum()),
                "n_switched": int(switch_mask.loc[cluster_mask].sum()),
                "percent_switched": round(float(100.0 * switch_mask.loc[cluster_mask].mean()), 2),
                "best_label_mapping": str(label_mapping),
                "best_label_match_count": int(best_match_count),
            }
        )

    crosstab_df = pd.crosstab(
        baseline_labels.rename("baseline_cluster"),
        cleaned_labels.rename("cleaned_cluster"),
    )
    for baseline_cluster_id in crosstab_df.index:
        for cleaned_cluster_id in crosstab_df.columns:
            crosstab_rows.append(
                {
                    "n_clusters": int(k),
                    "baseline_cluster": int(baseline_cluster_id),
                    "cleaned_cluster": int(cleaned_cluster_id),
                    "n_events": int(crosstab_df.loc[baseline_cluster_id, cleaned_cluster_id]),
                }
            )

    count_rows.extend(cleaned_counts_df.assign(n_clusters=int(k)).to_dict(orient="records"))

    cleaned_cluster_medians_df = (
        cleaned_feature_df.loc[cleaned_labels.index]
        .groupby(cleaned_labels)[CLEANED_FEATURE_COLUMNS]
        .median()
        .round(3)
        .reset_index()
        .rename(columns={cleaned_labels.name: "cluster_id"})
        .assign(n_clusters=int(k))
    )
    median_rows.extend(cleaned_cluster_medians_df.to_dict(orient="records"))

switching_df = pd.DataFrame(switching_rows)
crosstab_long_df = pd.DataFrame(crosstab_rows)
cleaned_cluster_counts_long_df = pd.DataFrame(count_rows)
cleaned_cluster_medians_long_df = pd.DataFrame(median_rows)
solution_summary_df = pd.DataFrame(solution_rows)

switching_df.to_csv(SWITCHING_PATH, index=False)
maybe_copy_to_drive(SWITCHING_PATH)
crosstab_long_df.to_csv(CROSSTAB_PATH, index=False)
maybe_copy_to_drive(CROSSTAB_PATH)
cleaned_cluster_counts_long_df.to_csv(COUNTS_PATH, index=False)
maybe_copy_to_drive(COUNTS_PATH)
cleaned_cluster_medians_long_df.to_csv(MEDIANS_PATH, index=False)
maybe_copy_to_drive(MEDIANS_PATH)
solution_summary_df.to_csv(SOLUTION_SUMMARY_PATH, index=False)
maybe_copy_to_drive(SOLUTION_SUMMARY_PATH)
variance_export_df.to_csv(VARIANCE_PATH, index=False)
maybe_copy_to_drive(VARIANCE_PATH)
loadings_export_df.to_csv(LOADINGS_PATH, index=False)
maybe_copy_to_drive(LOADINGS_PATH)
driver_export_df.to_csv(DRIVERS_PATH, index=False)
maybe_copy_to_drive(DRIVERS_PATH)
clustered_events_export_df.to_csv(CLUSTERED_EVENTS_PATH, index=False)
maybe_copy_to_drive(CLUSTERED_EVENTS_PATH)

print("Current versus cleaned low-level clustering solution summary")
display(solution_summary_df)
print("\nBaseline versus cleaned low-level silhouette comparison across k = 2, 3, 4")
display(quality_comparison_df)
print("\nBaseline and cleaned low-level quality scans across k = 2, 3, 4")
display(quality_scan_long_df)
print("\nSwitching summary after replacing the terrain-sensitive low-level features with their Russian-coastal-cleaned versions")
display(switching_df)
print("\nBaseline versus cleaned cluster cross-tabs across k = 2, 3, 4")
display(crosstab_long_df)
print("\nCleaned cluster counts by k")
display(cleaned_cluster_counts_long_df)
print("\nCleaned cluster medians by k")
display(cleaned_cluster_medians_long_df)
print("\nPCA variance comparison")
display(variance_export_df)
print("\nTop PCA loading drivers")
display(driver_export_df)


In [ ]:
feature_plot_specs = [
    (BASE_RATIO_COLUMN, CLEANED_RATIO_COLUMN, "Coastal/JPCZ signed-divergence ratio"),
    (BASE_SOJ_VORT_COLUMN, CLEANED_SOJ_VORT_COLUMN, "Sea of Japan mean 925 hPa vorticity"),
    (BASE_T850_COLUMN, CLEANED_T850_COLUMN, "Front-box maximum |grad T850|"),
]

fig, axes = plt.subplots(1, len(feature_plot_specs), figsize=(5.6 * len(feature_plot_specs), 5.8), constrained_layout=True)
if len(feature_plot_specs) == 1:
    axes = [axes]

for ax, (base_column, cleaned_column, title) in zip(axes, feature_plot_specs):
    plot_data = []
    positions = []
    tick_positions = []
    tick_labels = []
    colors = []
    for cluster_id in sorted(cleaned_event_feature_df["current_cluster_id"].astype(int).unique()):
        subset = cleaned_event_feature_df.loc[cleaned_event_feature_df["current_cluster_id"].astype(int) == cluster_id]
        cluster_positions = [cluster_id - 0.15, cluster_id + 0.15]
        positions.extend(cluster_positions)
        tick_positions.append(cluster_id)
        tick_labels.append(f"C{cluster_id}")
        plot_data.extend([
            subset[base_column].dropna().values,
            subset[cleaned_column].dropna().values,
        ])
        colors.extend(["#9ecae1", "#fdae6b"])
    box = ax.boxplot(plot_data, positions=positions, widths=0.22, patch_artist=True, showfliers=False)
    for patch, color in zip(box["boxes"], colors):
        patch.set_facecolor(color)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    ax.set_title(title)
    ax.set_ylabel(axis_label(cleaned_column))
    ax.grid(axis="y", alpha=0.25)
    if positions:
        ax.plot([], [], color="#9ecae1", linewidth=8, label="Unmasked")
        ax.plot([], [], color="#fdae6b", linewidth=8, label="Cleaned")
        ax.legend(loc="best", fontsize=9)

if SAVE_PLOTS:
    fig.savefig(FEATURE_BOXPLOT_PATH, dpi=180, bbox_inches="tight")
    maybe_copy_to_drive(FEATURE_BOXPLOT_PATH, verbose=False)
plt.show()

silhouette_fig, silhouette_ax = plt.subplots(figsize=(7.5, 5), constrained_layout=True)
silhouette_ax.plot(
    quality_comparison_df["n_clusters"],
    quality_comparison_df["mean_silhouette_score_baseline"],
    marker="o",
    linewidth=2,
    label="Baseline current features",
)
silhouette_ax.plot(
    quality_comparison_df["n_clusters"],
    quality_comparison_df["mean_silhouette_score_cleaned"],
    marker="o",
    linewidth=2,
    label="Russian-coastal cleaned low-level features",
)
silhouette_ax.axvline(recommended_k, color="0.5", linestyle="--", linewidth=1.2, label=f"Recommended cleaned k={recommended_k}")
silhouette_ax.set_xlabel("Number of clusters (k)")
silhouette_ax.set_ylabel("Mean silhouette score")
silhouette_ax.set_title("Baseline versus cleaned low-level silhouette comparison")
silhouette_ax.grid(alpha=0.25)
silhouette_ax.legend(loc="best")
if SAVE_PLOTS:
    silhouette_fig.savefig(SILHOUETTE_PLOT_PATH, dpi=180, bbox_inches="tight")
    maybe_copy_to_drive(SILHOUETTE_PLOT_PATH, verbose=False)
plt.show()

print("Saved cleaned low-level decision plots")
display(pd.DataFrame({
    "plot": [
        "Current-cluster boxplots for unmasked versus cleaned low-level features",
        "Baseline versus cleaned silhouette comparison",
    ],
    "path": [str(FEATURE_BOXPLOT_PATH), str(SILHOUETTE_PLOT_PATH)],
}))


### How to read the result

- If the cleaned low-level features still support the same preferred `k`, then the original grouping was robust to this Russian-coastal correction.
- If the cleaned-feature silhouette scan shifts clearly toward a different `k`, then the terrain-sensitive low-level inputs were helping define the earlier grouping framework.
- If one low-level feature changes strongly while another stays nearly unchanged, that tells us the Russian-coastal exclusion is affecting the features asymmetrically rather than simply shrinking everything at once.
- This notebook is the decision point before remaking the cleaned composite figures in the next notebook.
